# Precision-Recall (PR)

This code demonstrates how to use Precision-Recall (PR) curves to analyze binary classifiers and select an appropriate classification threshold using a synthetic dataset. Here's what the code does:

1. **Data Preparation and Model Training**
   - Creates a synthetic 2D binary classification dataset with class imbalance
   - Trains a Random Forest classifier
   - Calculates prediction probabilities for test data

2. **Precision-Recall Curve Analysis**
   - Generates the PR curve (Precision vs Recall)
   - Calculates Average Precision (AP)
   - Compares the model against the positive-class baseline

3. **Threshold Selection Methods**
   - Implements common threshold selection strategies for PR analysis:
     - **Best F1 score**: Maximizes the harmonic mean of precision and recall
     - **Target precision**: Finds the highest recall while maintaining a desired precision level

4. **Performance Evaluation**
   - Compares model performance at different thresholds
   - Calculates key metrics (accuracy, precision, recall, F1 score)
   - Visualizes confusion matrices for selected thresholds

5. **Visualization**
   - Displays the synthetic dataset in 2D
   - Plots the PR curve with selected threshold points
   - Shows how precision, recall, and F1 vary with threshold values
   - Displays confusion matrices for representative thresholds

When you run this code, you'll see:
- How precision and recall trade off as the threshold changes
- Which threshold maximizes F1
- How to enforce a minimum precision requirement when choosing a threshold


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_recall_curve, average_precision_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Set random seed for reproducibility
np.random.seed(42)

# Generate a synthetic 2D binary classification dataset with class imbalance
X, y = make_classification(
    n_samples=1500,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    n_clusters_per_class=1,
    class_sep=1.2,
    weights=[0.8, 0.2],
    flip_y=0.01,
    random_state=42
)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

# Train a Random Forest classifier
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

# Get probability scores for the positive class
y_scores = clf.predict_proba(X_test)[:, 1]

# Calculate Precision-Recall curve
precision, recall, thresholds = precision_recall_curve(y_test, y_scores)

# Average Precision summarizes the PR curve
avg_precision = average_precision_score(y_test, y_scores)
positive_rate = y_test.mean()

# Function to find useful thresholds for PR analysis
def find_optimal_thresholds(precision, recall, thresholds, min_precision=0.90):
    """
    Find useful thresholds for PR analysis:
    1. Threshold that maximizes F1 score
    2. Threshold that satisfies a minimum precision target while maximizing recall
    """
    precision_t = precision[:-1]
    recall_t = recall[:-1]

    f1_scores = 2 * precision_t * recall_t / np.clip(precision_t + recall_t, 1e-12, None)
    best_f1_idx = np.argmax(f1_scores)

    valid_precision = np.where(precision_t >= min_precision)[0]
    if len(valid_precision) > 0:
        target_idx = valid_precision[np.argmax(recall_t[valid_precision])]
    else:
        target_idx = best_f1_idx

    return {
        "best_f1": (thresholds[best_f1_idx], best_f1_idx),
        "target_precision": (thresholds[target_idx], target_idx),
        "min_precision": min_precision
    }

# Find representative thresholds
optimal_thresholds = find_optimal_thresholds(precision, recall, thresholds, min_precision=0.90)

# Evaluate model at different thresholds
def evaluate_threshold(y_true, y_scores, threshold):
    """Evaluate model performance at a given threshold"""
    y_pred = (y_scores >= threshold).astype(int)
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    specificity = tn / (tn + fp)

    return {
        "threshold": threshold,
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "specificity": specificity,
        "f1_score": f1,
        "confusion_matrix": {
            "TP": tp, "FP": fp,
            "TN": tn, "FN": fn
        }
    }

# Evaluate different thresholds
default_results = evaluate_threshold(y_test, y_scores, 0.5)
best_f1_results = evaluate_threshold(y_test, y_scores, optimal_thresholds["best_f1"][0])
target_precision_results = evaluate_threshold(y_test, y_scores, optimal_thresholds["target_precision"][0])

# Visualize the dataset and PR analysis
plt.figure(figsize=(16, 8))

# Plot the synthetic dataset
plt.subplot(2, 3, 1)
plt.scatter(X_train[y_train == 0, 0], X_train[y_train == 0, 1], alpha=0.5, s=20, label='Negative class')
plt.scatter(X_train[y_train == 1, 0], X_train[y_train == 1, 1], alpha=0.7, s=20, label='Positive class')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Synthetic Training Dataset')
plt.legend(loc='best')
plt.grid(True, alpha=0.3)

# Plot PR curve
plt.subplot(2, 3, 2)
plt.plot(recall, precision, color='blue', lw=2, label=f'PR curve (AP = {avg_precision:.3f})')
plt.axhline(y=positive_rate, color='gray', linestyle='--', label=f'Baseline = {positive_rate:.3f}')

# Mark representative points
best_f1_idx = optimal_thresholds["best_f1"][1]
target_precision_idx = optimal_thresholds["target_precision"][1]

plt.scatter(recall[best_f1_idx], precision[best_f1_idx], marker='o', color='red',
            label=f"Best F1 (threshold={optimal_thresholds['best_f1'][0]:.3f})")
plt.scatter(recall[target_precision_idx], precision[target_precision_idx], marker='s', color='green',
            label=f"Precision >= {optimal_thresholds['min_precision']:.2f} (threshold={optimal_thresholds['target_precision'][0]:.3f})")

# Default threshold of 0.5
default_idx = np.abs(thresholds - 0.5).argmin()
plt.scatter(recall[default_idx], precision[default_idx], marker='^', color='purple',
            label='Default (threshold=0.5)')

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve with Selected Thresholds')
plt.legend(loc='lower left')
plt.grid(True, alpha=0.3)

# Plot threshold vs various metrics
plt.subplot(2, 3, 3)
metrics_at_thresholds = []
sampled_thresholds = np.linspace(0, 1, 100)
for threshold in sampled_thresholds:
    metrics = evaluate_threshold(y_test, y_scores, threshold)
    metrics_at_thresholds.append(metrics)

# Extract metrics for plotting
precisions = [m['precision'] for m in metrics_at_thresholds]
recalls = [m['recall'] for m in metrics_at_thresholds]
f1_scores = [m['f1_score'] for m in metrics_at_thresholds]
accuracies = [m['accuracy'] for m in metrics_at_thresholds]

plt.plot(sampled_thresholds, precisions, label='Precision')
plt.plot(sampled_thresholds, recalls, label='Recall')
plt.plot(sampled_thresholds, f1_scores, label='F1 Score')
plt.plot(sampled_thresholds, accuracies, label='Accuracy', alpha=0.7)

# Mark selected thresholds
plt.axvline(x=optimal_thresholds['best_f1'][0], color='red', linestyle='--', alpha=0.5,
            label='Best F1 threshold')
plt.axvline(x=optimal_thresholds['target_precision'][0], color='green', linestyle='--', alpha=0.5,
            label='Target precision threshold')
plt.axvline(x=0.5, color='purple', linestyle='--', alpha=0.5, label='Default threshold')

plt.xlabel('Threshold')
plt.ylabel('Metric Value')
plt.title('Performance Metrics vs. Threshold')
plt.legend(loc='center left')
plt.grid(True, alpha=0.3)

# Plot the confusion matrices for different thresholds
def plot_confusion_matrix(ax, results, title):
    cm = np.array([
        [results['confusion_matrix']['TN'], results['confusion_matrix']['FP']],
        [results['confusion_matrix']['FN'], results['confusion_matrix']['TP']]
    ])
    ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    ax.set_title(title)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['Negative', 'Positive'])
    ax.set_yticklabels(['Negative', 'Positive'])

    # Add text annotations
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha='center', va='center',
                    color='white' if cm[i, j] > cm.max()/2 else 'black')

# Plot confusion matrices for each threshold
plt.subplot(2, 3, 4)
plot_confusion_matrix(plt.gca(), default_results, f"Default Threshold (0.5)\nF1: {default_results['f1_score']:.3f}")

plt.subplot(2, 3, 5)
plot_confusion_matrix(plt.gca(), best_f1_results, f"Best F1 Threshold ({optimal_thresholds['best_f1'][0]:.3f})\nF1: {best_f1_results['f1_score']:.3f}")

plt.subplot(2, 3, 6)
plot_confusion_matrix(plt.gca(), target_precision_results, f"Precision >= {optimal_thresholds['min_precision']:.2f}\nThreshold: {optimal_thresholds['target_precision'][0]:.3f}")

plt.tight_layout()

# Print summary of results
print('Comparison of Thresholds:')
print('-' * 50)
print('Default (0.5):')
for k, v in default_results.items():
    if k != 'confusion_matrix':
        print(f'  {k}: {v:.4f}')

print(f"\nThreshold that maximizes F1: {optimal_thresholds['best_f1'][0]:.4f}")
print(f"Best F1 score: {best_f1_results['f1_score']:.4f}")
for k, v in best_f1_results.items():
    if k != 'confusion_matrix':
        print(f'  {k}: {v:.4f}')

print(f"\nThreshold with precision >= {optimal_thresholds['min_precision']:.2f}: {optimal_thresholds['target_precision'][0]:.4f}")
for k, v in target_precision_results.items():
    if k != 'confusion_matrix':
        print(f'  {k}: {v:.4f}')

# Show the plots
plt.show()
